In [1]:
# Remove doublings from table and estimate them
# FM 20/10/2025

In [ ]:
import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo

import pandas as pd
import numpy as np
from collections import defaultdict

/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-10-20 19:36:54.976692: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
fileorch='midis/liszt_classical_archives-0/orchestra.mid'

In [4]:
print(f"Orchestra file: {fileorch}")
dforch = midi_to_dataframe(fileorch)
dforch = dforch.sort_values(
    ['onset in quarter notes', 'duration in quarter notes', 'track number'],
    ascending=[True, True, True]
)
print(dforch.columns)
print("Table:\n", dforch)
np_orch = dforch.to_numpy()
mapping = learn_quaterna_mapping(np_orch, ytarget="track-channel")
print("Mapping:", mapping)

Orchestra file: midis/liszt_classical_archives-0/orchestra.mid
Index(['track number', 'track name', 'channel', 'program',
       'onset in quarter notes', 'duration in quarter notes', 'pitch',
       'velocity'],
      dtype='object')
Table:
       track number   track name  channel  program  onset in quarter notes  \
0                1        Flute        1       73                     0.0   
1                1        Flute        1       73                     0.0   
585              2         Oboe        2       68                     0.0   
586              2         Oboe        2       68                     0.0   
1206             3     Clarinet        3       71                     0.0   
...            ...          ...      ...      ...                     ...   
1205             2         Oboe        2       68                  1574.0   
4791            10       Violin       10       40                  1574.0   
5491            11        Viola       11       41               

In [5]:
def reduce_df_with_transform(df, tol=1.0, transformations=None):
    df_reduced = df
    df_hashed = defaultdict(list)
    count_match = {
        'transformed': 0,
        'direct': 0,
        'none': 0
    }
    df_reduced["doubled"] = 0
    for func, kwargs in transformations:
        df_reduced[f"{func.__name__}_{kwargs}"] = 0

    todrop = []

    for idx, note in df_reduced.iterrows():
        key = (note['onset in quarter notes'], note['pitch'])        

        matched = False
        for candidate_match_index in df_hashed[key]:
            candidate_match = df_reduced.loc[candidate_match_index]
            dur_diff = abs(candidate_match['duration in quarter notes'] - note['duration in quarter notes'])
            if dur_diff <= note['duration in quarter notes'] * tol:
                df_reduced.at[candidate_match_index, "doubled"] += 1
                matched = True
                count_match['direct'] += 1
                todrop.append(idx)
                break

        if transformations and not matched:
            for func, kwargs in transformations:
                if not matched:
                    transformed = func(note, **kwargs)
                    for t in transformed:
                        key_t = (t['onset in quarter notes'], t['pitch'])
                        if len(df_hashed[key_t]) > 0: # if something matches with the transformation
                            for candidate_match_index in df_hashed[key_t]:
                                candidate_match = df_reduced.loc[candidate_match_index]
                                dur_diff = abs(candidate_match['duration in quarter notes'] - note['duration in quarter notes'])
                                if dur_diff <= note['duration in quarter notes'] * tol:
                                    df_reduced.at[candidate_match_index, f"{func.__name__}_{kwargs}"] += 1
                                    matched = True
                                    count_match['transformed'] += 1
                                    todrop.append(idx)
                                    break

        if not matched:
            df_hashed[key].append(idx)
            count_match['none'] += 1

    print(count_match)
    print(f"Original size: {df_reduced.shape}")
    print(f"Dropping {todrop}")
    df_reduced.drop(todrop, inplace=True)
    print(f"Size after drop: {df_reduced.shape}")

    return df_reduced


In [6]:
def transpose(note, inverse=False, n_semitones=12):
    # Example: transpose pitch
    if inverse:
        n_semitones = - n_semitones
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [7]:
def split_duration(note,  inverse=False, smallest_unit=1.0):
    # Split a note into a set of notes of duration smallest_unit
    # ['track number', 'track name', 'channel', 'program', 'onset in quarter notes', 'duration in quarter notes', 'pitch', 'velocity']
    if inverse:
        raise SyntaxError("TODO: implement iverse for split_duration")
    transformed_notes = []
    dur = note['duration in quarter notes']
    n = int(dur // smallest_unit)
    for i in range(n):
        new_note = note.copy()
        new_note['onset in quarter notes'] = note['onset in quarter notes'] + smallest_unit*i
        new_note['duration in quarter notes'] = smallest_unit
        transformed_notes.append(new_note)
    if dur - smallest_unit*n > 0:
        new_note = note.copy()
        new_note['onset in quarter notes'] = note['onset in quarter notes'] + smallest_unit*n
        new_note['duration in quarter notes'] = dur - smallest_unit*n
        transformed_notes.append(new_note)
    return transformed_notes

In [8]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]


In [9]:
df_reduced = reduce_df_with_transform(dforch, tol=0.2, transformations=transformations)

{'transformed': 2522, 'direct': 848, 'none': 3247}
Original size: (6617, 15)
Dropping [585, 586, 1207, 1676, 2775, 2776, 2777, 3775, 3776, 4792, 5492, 6081, 2370, 587, 588, 1208, 1209, 1678, 2778, 2779, 2780, 3777, 3778, 4793, 5493, 6082, 589, 590, 1211, 1680, 2371, 2372, 2781, 2782, 2783, 3779, 3780, 3781, 4794, 5494, 6083, 591, 592, 1212, 1213, 1682, 2374, 2784, 2785, 2786, 3782, 3783, 3784, 4795, 5495, 6084, 593, 594, 1215, 1684, 2375, 2787, 2788, 2789, 3785, 3786, 4796, 5496, 6085, 595, 596, 1217, 1686, 2376, 2790, 2791, 2792, 3787, 3788, 4797, 5497, 6086, 597, 598, 1219, 1688, 2377, 2793, 2794, 2795, 3789, 3790, 4798, 5498, 6087, 599, 600, 1220, 1221, 1690, 2378, 2628, 2629, 2796, 2797, 2798, 3791, 3792, 3793, 4799, 5499, 6088, 3794, 3795, 3796, 3797, 5500, 3798, 2381, 601, 1222, 5501, 3799, 3800, 1693, 5502, 3801, 5503, 1696, 3802, 3803, 5504, 3804, 1698, 3805, 3806, 5505, 3807, 5506, 3808, 5507, 3809, 5508, 1703, 2382, 2383, 3810, 1223, 1224, 5509, 1705, 2384, 2385, 3811, 3812, 

In [10]:
dforch

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,doubled,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
0,1,Flute,1,73,0.0,2.0,82,60,0,3,0,0,0,0,0
1,1,Flute,1,73,0.0,2.0,88,60,0,2,1,0,0,0,0
1206,3,Clarinet,3,71,0.0,2.0,67,60,1,1,0,0,0,0,0
1675,4,Bassoon,4,70,0.0,2.0,48,60,1,0,0,0,3,0,0
2369,5,Horn,5,60,0.0,3.0,48,60,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4789,10,Violin,10,40,1570.0,4.0,60,60,0,0,0,0,0,0,0
4790,10,Violin,10,40,1570.0,4.0,68,60,0,0,0,0,0,0,0
1204,2,Oboe,2,68,1574.0,1.0,75,60,0,1,0,0,0,0,0
1205,2,Oboe,2,68,1574.0,1.0,79,60,0,1,0,0,0,0,0


In [11]:
df_reduced

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,doubled,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
0,1,Flute,1,73,0.0,2.0,82,60,0,3,0,0,0,0,0
1,1,Flute,1,73,0.0,2.0,88,60,0,2,1,0,0,0,0
1206,3,Clarinet,3,71,0.0,2.0,67,60,1,1,0,0,0,0,0
1675,4,Bassoon,4,70,0.0,2.0,48,60,1,0,0,0,3,0,0
2369,5,Horn,5,60,0.0,3.0,48,60,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4789,10,Violin,10,40,1570.0,4.0,60,60,0,0,0,0,0,0,0
4790,10,Violin,10,40,1570.0,4.0,68,60,0,0,0,0,0,0,0
1204,2,Oboe,2,68,1574.0,1.0,75,60,0,1,0,0,0,0,0
1205,2,Oboe,2,68,1574.0,1.0,79,60,0,1,0,0,0,0,0


In [12]:
df_reduced[df_reduced["doubled"] > 0]

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,doubled,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
1206,3,Clarinet,3,71,0.0,2.0,67,60,1,1,0,0,0,0,0
1675,4,Bassoon,4,70,0.0,2.0,48,60,1,0,0,0,3,0,0
1677,4,Bassoon,4,70,2.0,1.0,41,60,1,0,0,0,3,0,0
4,1,Flute,1,73,4.0,2.0,77,60,1,3,0,0,0,0,0
1210,3,Clarinet,3,71,4.0,2.0,62,60,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2356,4,Bassoon,4,70,1395.5,3.5,47,60,1,0,0,0,0,0,0
2357,4,Bassoon,4,70,1401.0,0.5,43,60,1,0,0,0,0,0,0
576,1,Flute,1,73,1404.0,6.0,75,60,1,1,0,0,0,0,0
1194,2,Oboe,2,68,1404.0,6.0,72,60,1,2,3,0,0,0,0


In [13]:
df_reduced[df_reduced["transpose_{'n_semitones': 46}"] > 0]

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,doubled,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46}
62,1,Flute,1,73,44.0,2.0,89,60,0,1,0,1,0,0,0
329,1,Flute,1,73,419.0,1.0,84,60,0,2,1,1,0,0,0


In [14]:
save_midi_with_exact_timing_structure(df_reduced, 'midis/example_reduced.mid', reference_midi_path=fileorch)


=== SAVING WITH EXACT TIMING STRUCTURE ===
Extracting timing structure from midis/liszt_classical_archives-0/orchestra.mid
ticks_per_beat: 1024
Found 0 timing events:
Using ticks_per_beat: 1024
Applying key signature fix for MuseScore compatibility...
✓ Added consistent key signatures to 10 instrument tracks
✅ Saved midis/example_reduced.mid with exact timing structure preserved
   - 0 timing events preserved
   - 10 instrument tracks created
   - Key signatures fixed for MuseScore compatibility


In [15]:
from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import numpy as np
import time
import joblib

KERAS_AVAILABLE = False

def estimate_transform(df, ytarget="transpose_{'n_semitones': 12}", model="XGBoost", pipeline_path=""):
    """
    FM
    """
    
    # Define available classifiers and their names
    classifiers_map = {
        "XGBoost": XGBClassifier(),
        "RandomForest": RandomForestClassifier(),
        "DecisionTree": DecisionTreeClassifier(),
        "NearestNeighbors": KNeighborsClassifier(1),
        "MLP3": MLPClassifier(hidden_layer_sizes=(128, 128, 128)),
        "NaiveBayes": GaussianNB(),
        "MLP1": MLPClassifier(),
        "AdaBoost": AdaBoostClassifier(),
    }
    
    if KERAS_AVAILABLE:
        classifiers_map["LSTMClassifier"] = KerasClassifierWrapper(build_lstm_classifier)
        classifiers_map["TransformerClassifier"] = KerasClassifierWrapper(build_transformer_classifier)

    # Check if the requested model is available
    if model not in classifiers_map:
        if "LSTMClassifier" in model or "TransformerClassifier" in model:
            print(f"Error: Keras is not available. Cannot use {model}.")
            return
        else:
            print(f"Error: Invalid model name '{model}'. Available models are: {list(classifiers_map.keys())}")
            return
            
    clf_name = model
    clf = classifiers_map[clf_name]

    # Load and process source file
    #print(f"Learning {ytarget}")
    df = df.sort_values(
        ['onset in quarter notes', 'duration in quarter notes', 'track number'],
        ascending=[True, True, True]
    )
    nmat = df.to_numpy()
    mapping = learn_quaterna_mapping(nmat, ytarget)
    #print("Mapping:", mapping)
    
    X, _ = defineXy(nmat, ytarget)
    y = df[ytarget].to_numpy()
    print("Labels", np.unique(y))
    print("Number of events:", X.shape[0])
    print("Last onset at", X[X.shape[0] - 1, 0])

    if len(np.unique(y)) == 1:
        print("Only one label in target variable")
        return
    
    # Partition the dataset
    X_train, X_test, y_train, y_test, le = split_and_encode(X, y, test_size=0.2, random_state=42)
    X_train_f, _, y_train_f, _, le_f = split_and_encode(X, y, test_size=0, random_state=42)
    
    # Train and predict with the specified classifier
    print(f"\n--------- {clf_name} ---------")
    start = time.time()
    
    if clf_name in ["LSTMClassifier", "TransformerClassifier"]:
        clf_pipeline = make_pipeline(StandardScaler(), clf)
    else:
        clf_pipeline = clf
        
    clf_pipeline.fit(X_train, y_train)
    score = clf_pipeline.score(X_test, y_test) # This is accuracy (?)
    # TODO: Here and everywhere, use more scoring functions (prec, rec, f1, acc)
    end = time.time()
    
    print("Train Time (sec):", f"{end - start:.4f}")
    print("Score on Test Set (20% split):", f"{score:.4f}")
    #
    # Save all needed artifacts
    if pipeline_path!="": 
        artifact = {
            "pipeline": clf_pipeline,                 # pipeline
            "label_encoder": le_f,           # label encoder for final training
            "mapping": mapping,              # quaterna reconstruction mapping
            "ytarget": ytarget,              # 
            }

        joblib.dump(artifact, pipeline_path)
        print(f"[AMO-XGB SAVE] Pipeline saved: {pipeline_path}")
    #

In [16]:
for func, kwargs in transformations:
    ytarget = f"{func.__name__}_{kwargs}"
    #print(f"Learning {ytarget}")
    print(df_reduced[ytarget].value_counts(normalize=True))
    estimate_transform(df_reduced, ytarget=ytarget, model="XGBoost", pipeline_path="")
    print("\n")

# TODO: Use one model for multi-variate target prediction

transpose_{'n_semitones': 12}
0    0.639359
1    0.263936
2    0.058208
3    0.029258
4    0.008623
5    0.000616
Name: proportion, dtype: float64
Labels [0 1 2 3 4 5]
Number of events: 3247
Last onset at 1574.0
y_train, test size: 0.2 , labels: [0 1 2 3 4 5]
y_train, test size: 0 , labels: [0 1 2 3 4 5]

--------- XGBoost ---------
Train Time (sec): 0.9589
Score on Test Set (20% split): 0.8123


transpose_{'n_semitones': 24}
0    0.874654
1    0.081306
2    0.033877
3    0.008315
4    0.001540
5    0.000308
Name: proportion, dtype: float64
Labels [0 1 2 3 4 5]
Number of events: 3247
Last onset at 1574.0
y_train, test size: 0.2 , labels: [0 1 2 3 4 5]
y_train, test size: 0 , labels: [0 1 2 3 4 5]

--------- XGBoost ---------
Train Time (sec): 0.8140
Score on Test Set (20% split): 0.9123


transpose_{'n_semitones': 46}
0    0.999384
1    0.000616
Name: proportion, dtype: float64
Labels [0 1]
Number of events: 3247
Last onset at 1574.0
y_train, test size: 0.2 , labels: [0 1]
y_train, tes

In [17]:
def expand_estimated_transform(df, transformations=None):
    """
    Expands a reduced dataframe by recreating rows that were removed 
    due to direct duplicates or transformations in `reduce_df_with_transform`.

    Parameters
    ----------
    df : pd.DataFrame
        The reduced dataframe returned by reduce_df_with_transform.
    transformations : list of tuples, optional
        List of (func, kwargs) transformation specifications.
        Each func must accept and return a dict representing a note.

    Returns
    -------
    pd.DataFrame
        Expanded dataframe including the recreated duplicated and transformed rows.
    """
    expanded_rows = []

    # iterate through all remaining notes
    for idx, row in df.iterrows():
        note_dict = row.to_dict()
        expanded_rows.append(note_dict)  # always keep original

        # Handle doublings (even without transformations)
        if "doubled" in df.columns and row["doubled"] > 0:
            for _ in range(int(row["doubled"])):
                expanded_rows.append(note_dict.copy())

        # Handle transformations if available
        if transformations:
            for func, kwargs in transformations:
                col_name = f"{func.__name__}_{kwargs}"
                if col_name in df.columns and row[col_name] > 0:
                    for _ in range(int(row[col_name])):
                        # Apply the inverse transformation
                        try:
                            transformed_notes = func(row, inverse=True, **kwargs)
                        except TypeError as err:
                            raise TypeError(f"Function {func.__name__} is not invertible! {err}")

                        # func returns a list of transformed notes
                        for t in transformed_notes:
                            expanded_rows.append(t)

    # Rebuild dataframe
    df_expanded = pd.DataFrame(expanded_rows).reset_index(drop=True)
    return df_expanded

    

In [18]:
# Example
df_reconstructed = expand_estimated_transform(df_reduced, transformations=transformations)
df_reconstructed

,track number,track name,channel,program,onset in quarter notes,duration in quarter notes,pitch,velocity,doubled,transpose_{'n_semitones': 12},transpose_{'n_semitones': 24},transpose_{'n_semitones': 46},transpose_{'n_semitones': -12},transpose_{'n_semitones': -24},transpose_{'n_semitones': -46},base_instrument
0,1,Flute,1,73,0.0,2.0,82,60,0,3,0,0,0,0,0,Flute
1,1,Flute,1,73,0.0,2.0,70,60,0,3,0,0,0,0,0,Flute
2,1,Flute,1,73,0.0,2.0,70,60,0,3,0,0,0,0,0,Flute
3,1,Flute,1,73,0.0,2.0,70,60,0,3,0,0,0,0,0,Flute
4,1,Flute,1,73,0.0,2.0,88,60,0,2,1,0,0,0,0,Flute
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6612,2,Oboe,2,68,1574.0,1.0,63,60,0,1,0,0,0,0,0,Oboe
6613,2,Oboe,2,68,1574.0,1.0,79,60,0,1,0,0,0,0,0,Oboe
6614,2,Oboe,2,68,1574.0,1.0,67,60,0,1,0,0,0,0,0,Oboe
6615,12,Violoncello,12,42,1574.0,1.0,46,60,0,1,0,0,0,0,0,Violoncello
